# Day 57 — Security, data privacy & ethical considerations
Objectives:
- Identify PII and sensitive attributes.
- Basic de-identification and minimization strategies.
- Ethical considerations, fairness checks, and documentation.
Note: This notebook demonstrates lightweight checks; real programs require legal/policy review.

In [ ]:
import re, pandas as pd
from pathlib import Path
sample = pd.DataFrame({
    'name': ['Alice Smith','Bob Jones'],
    'email': ['alice@example.com','bob@company.org'],
    'phone': ['+1-415-555-1212','(212) 555-9898'],
    'notes': ['Met on 2025-01-01','Lives near 5th Ave']
})
sample


## PII detection (simple regex demo)
Caution: regex is imperfect; use specialized tools for robust detection.

In [ ]:
EMAIL_RE = re.compile(r'[\w.%-]+@[\w.-]+\.[A-Za-z]{2,}')
PHONE_RE = re.compile(r'(?:\+?\d{1,3}[-.\s]?)?(?:\(\d{3}\)|\d{3})[-.\s]?\d{3}[-.\s]?\d{4}')
def find_pii(s: str) -> dict:
    emails = EMAIL_RE.findall(s)
    phones = PHONE_RE.findall(s)
    return {'emails': emails, 'phones': phones}

sample['pii'] = sample.apply(lambda r: {
    'email': EMAIL_RE.findall(r['email']),
    'phone': PHONE_RE.findall(r['phone']),
    'notes': find_pii(r['notes'])
}, axis=1)
sample[['pii']]


## De-identification strategies
- Remove direct identifiers (name, email, phone).
- Pseudonymize with stable hashes.
- Mask partial values.
- Limit retention and access (data minimization).
Below: pseudonymize names with a salted hash.

In [ ]:
import hashlib
SALT = b'secret-salt'  # store securely via environment vars/secret manager
def pseudo(value: str) -> str:
    h = hashlib.sha256(SALT + value.encode()).hexdigest()[:10]
    return f'id_{h}'

redacted = sample.copy()
redacted['name_pseudo'] = redacted['name'].map(pseudo)
redacted = redacted.drop(columns=['name','email','phone'])
redacted


## How to use this notebook

Select the `Python (ds60sqlpy)` kernel, start at the first cell, and
write each prediction before execution. Keep attempts in the
provided scratch cell or new cells. Restart the kernel and run from
the top before calling the work reproducible.

## Concept lab — data minimization, privacy boundaries, fairness evidence, and accountable controls

### Mental model

Security asks how systems resist misuse; privacy asks whether collection,
use, access, retention, and disclosure of data are justified; ethics
asks who benefits or is harmed and how decisions remain accountable.
These overlap but are not interchangeable.

Direct identifiers can name a person; quasi-identifiers can re-identify
in combination. Redaction removes visibility, while pseudonymization
retains linkability and is therefore still personal data. Fairness
metrics require a relevant decision, group definitions, denominators,
uncertainty, and harm analysis—not a single parity number.

### Read the API before running it

- **allowlist output fields:** starts from data that is necessary rather than trying to detect every sensitive field after collection.
- **detection → review → action:** treats regex/PII scanners as bounded signals with false positives and false negatives.
- **group metric + support:** reports numerator/denominator and uncertainty so tiny groups do not create confident-looking claims.

For every call, identify input data, learned state, returned value,
and a check that can fail. That habit prevents a successful cell
from being mistaken for a correct analysis.

### Focused example A — minimize a record through an allowlist

**Predict first:** write down the expected shape, type, ordering, or
direction of the result. Then run the next cell.

**Assumption:** The purpose and access model justify each allowed field, including the opaque record ID.

In [ ]:
raw_record = {
    "record_id": "row-17",
    "email": "learner@example.invalid",
    "age_band": "35-44",
    "prediction": 0.72,
    "free_text": "not required for this report",
}
allowed_fields = {"record_id", "age_band", "prediction"}
minimized = {key: raw_record[key] for key in allowed_fields}
print(minimized)
assert "email" not in minimized and "free_text" not in minimized

**Expected observation:** Only fields required for the stated report cross the boundary; unnecessary raw text and contact data do not.

Do not force exact equality for estimates based on samples. Record
the seed, sample size, tolerance, and metric where they matter.

### Focused example B — attach sample support to group error rates

This example changes one important condition. Predict how and why
the result should differ from Example A.

**Assumption:** The group definitions are lawful, meaningful for the decision, and measured consistently.

In [ ]:
groups = {
    "group_a": {"false_negatives": 8, "actual_positives": 80},
    "group_b": {"false_negatives": 1, "actual_positives": 5},
}
report = {
    name: {
        "false_negative_rate": values["false_negatives"] / values["actual_positives"],
        "support": values["actual_positives"],
    }
    for name, values in groups.items()
}
print(report)
assert report["group_b"]["support"] == 5

**Expected observation:** Group B's 20% rate rests on only five positive cases, so uncertainty and collection context are central.

### Debugging and practice ramp

**Common mistake:** Calling hashed identifiers anonymous, treating a regex scan as complete protection, or optimizing parity without decision context.

**Diagnostic:** Build a data-flow inventory: source, purpose, lawful/ethical basis, fields, access, retention, logs/artifacts, deletion, owners, and incident route.

| Stage | Action | Evidence |
|---|---|---|
| Recall | Define data minimization, privacy boundaries, fairness evidence, and accountable controls in your own words and identify its input and output. | A definition that does not rely on the library name. |
| Predict | Predict the examples before execution, including shape and direction. | A written prediction and an explanation of any mismatch. |
| Implement | Recreate one example with a changed but valid input. | Code plus an assertion for the central invariant. |
| Debug | Trigger the named mistake or edge case intentionally. | The observed symptom and the smallest diagnostic that isolates it. |
| Transfer | Apply the idea to a different local dataset or decision. | A stated assumption, metric, and reason the method is suitable. |

**Stop condition:** Do not expose real sensitive data in a lesson, log, screenshot, model artifact, prompt, or group report without explicit authorization and minimization.

Continue to the numbered practice only after you can explain both
examples without rereading their code.

## Fairness and bias (brief)
- Check subgroup performance metrics (e.g., by sex, race, age bucket).
- Avoid using protected attributes directly unless justified; document rationale.
- Consider disparate impact, calibration across groups.
- Provide model cards and data statements.

## Learner exercises and progressive hints

1. Build a DataFrame PII scanner covering column names and free text.

**Verify:** For task `Build a DataFrame PII scanner covering column names and free text`, record the exact command/input, terminal result or returned value, and repeat the critical check from a clean process or fresh state.






2. Add a function that masks email addresses and phone numbers in text.

**Verify:** For task `Add a function that masks email addresses and phone numbers in text`, demonstrate the concrete requirement “2. Add a function that masks email addresses and phone numbers in text” with explicit inputs, observable output, and one counterexample.






3. Simulate subgroup precision and recall for a classifier and compare groups.

**Verify:** For task `Simulate subgroup precision and recall for a classifier and compare groups`, record the seed, resampling unit, run count, estimate, and an analytic or hand-worked comparison with a stated tolerance; then assert the return type/shape/value for the stated valid input and assert the named boundary or invalid input raises/returns exactly the documented behavior.






4. Draft a one-page data-ethics checklist for your project.

**Verify:** For task `Draft a one-page data-ethics checklist for your project`, produce the requested artifact with every named field/control and walk one allowed plus one rejected scenario through it.







### Progressive hints

1. Return finding type, row identifier, column, and a safe count—do not log the
   raw matched value. Test false-positive and missed-format cases.
2. Preserve only the minimum structure needed for debugging and ensure repeated
   substitutions do not reveal the original.
3. Include group support and positive-label counts beside metrics. Avoid a
   conclusion when groups are too small for a stable estimate.
4. Name intended use, excluded use, affected people, owners, data rights,
   retention, access, monitoring, appeals, and incident response.

### Additional mastery practice

Combine data minimization, access control, threat modeling, privacy limits, fairness uncertainty, and incident response. Detection or masking alone is not protection.

Predict or plan before you run code. Use the hint only after an honest
attempt, and record the evidence that would prove your result correct.

5. **Threat modeling:** Create a data-flow diagram for collection, notebook, artifacts, API, logs, and backups. For each boundary, identify asset, actor, threat, control, residual risk, and owner.
   **Progressive hint:** Include accidental exposure and insider misuse, not only external attackers. Trace data copies and retention through every stage.

**Verify:** For task `Threat modeling: Create a data-flow diagram for collection, notebook, artifacts, API, logs, a...`, report row/feature shapes, seed/splitter, train-versus-validation evidence, and the metric used without consulting final-test labels; then verify identity/hash and metadata, then reload or inspect the artifact outside the creating state and test one tampered mismatch.







6. **Re-identification reasoning:** Generalize a small dataset to satisfy a chosen k-anonymity target, then demonstrate why k-anonymity does not prevent attribute disclosure or attacks using outside information.
   **Progressive hint:** Group quasi-identifiers, inspect equivalence-class sizes and sensitive value diversity, and measure utility loss.

**Verify:** For task `Re-identification reasoning: Generalize a small dataset to satisfy a chosen k-anonymity targe...`, assert the return type/shape/value for the stated valid input and assert the named boundary or invalid input raises/returns exactly the documented behavior; then state one precise claim, the evidence supporting it, the governing assumption, and a counterexample or limitation.







7. **Fairness uncertainty:** Bootstrap subgroup precision and recall, show confidence intervals and support, and compare a gap with a ratio. Explain what to do when one group's denominator is nearly zero.
   **Progressive hint:** Resample at the independent entity level when rows repeat. Undefined metrics should remain undefined rather than being forced to zero.

**Verify:** For task `Fairness uncertainty: Bootstrap subgroup precision and recall, show confidence intervals and...`, record the seed, resampling unit, run count, estimate, and an analytic or hand-worked comparison with a stated tolerance; then use identical data, split, metric, and budget for both sides; record a side-by-side result and isolate the condition that changed.







8. **Incident response:** Simulate discovering raw emails in a committed notebook output. Write the containment, notification, credential review, history cleanup decision, verification, and prevention steps.
   **Progressive hint:** Preserve a restricted incident record, stop further sharing, and assume copied history may exist. Redaction from the latest commit alone is insufficient.

**Verify:** For task `Incident response: Simulate discovering raw emails in a committed notebook output. Write the...`, record the seed, resampling unit, run count, estimate, and an analytic or hand-worked comparison with a stated tolerance; then produce the requested artifact with every named field/control and walk one allowed plus one rejected scenario through it.






Before opening the reference solution, explain the relevant assumption,
failure mode, and validation check for every answer.

In [ ]:
# Expanded mastery lab scratch space
#
# Keep the official solution closed until you have attempted each task.
# Add small assertions, shape checks, or metric comparisons as evidence.

# Practice 5 — Threat modeling


# Practice 6 — Re-identification reasoning


# Practice 7 — Fairness uncertainty


# Practice 8 — Incident response
